In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random
random.seed(2)

# Parameter Setting:

Teams = ['RRU','ERT','CIRS']
base_cost = {'RRU': 1, 'ERT':2, 'CIRS': 5}
delta_functionality = {'RRU':0.3, 'ERT': 0.55, 'CIRS': 0.75}
ServiceTime = {'RRU': 1, 'ERT':1, 'CIRS': 1} # Days
alpha = 0.5 #Adjustment Parameter

G = nx.DiGraph()
n = 6
cities = [f"C{i}" for i in range(1, n+1)]
def biased_choice(p_one):
    return 1 if random.random() >= p_one else 0

for city in cities:
    G.add_node(city)
    G.nodes[city]['Depot'] = biased_choice(0.90)
# (G.nodes[bridge_node]['Start'] + random.randint(4,7))
for i in range(len(cities)):
    for j in range(i + 1, len(cities)): 
        city1, city2 = cities[i], cities[j]
        num_bridges = random.randint(1, 1)
        prev_node = city1
        for b in range(num_bridges):
            bridge_node =f"B{city1}{city2}{b}"  
            G.add_node(bridge_node)
            G.nodes[bridge_node]['Start'] =random.randint(0,2)
            G.nodes[bridge_node]['BFI'] = round(random.uniform(0.2,0.4),2)
            G.nodes[bridge_node]['Due'] = (G.nodes[bridge_node]['Start'] + random.randint(2,5))
            G.nodes[bridge_node]['cost'] = {}
            G.nodes[bridge_node]['NewBFI'] = {}
          
            for team in Teams:
                G.nodes[bridge_node]['cost'][team] = round(base_cost[team]*(1+alpha*(1-G.nodes[bridge_node].get('BFI'))),2)
                G.nodes[bridge_node]['NewBFI'][team] = min(round(G.nodes[bridge_node].get('BFI') + delta_functionality[team],2),1)
                
            G.add_edge(prev_node, bridge_node)
            G.add_edge(bridge_node, prev_node)  
            
            speed = random.choice([60, 80, 120])  
            capacity = random.choice([500, 700, 800])  
            length = random.randint(1, 6)  
           
            G[prev_node][bridge_node]['speed'] = speed
            G[prev_node][bridge_node]['capacity'] = capacity
            G[prev_node][bridge_node]['length'] = length

            
            G[bridge_node][prev_node]['speed'] = speed
            G[bridge_node][prev_node]['capacity'] = capacity
            G[bridge_node][prev_node]['length'] = length
            prev_node = bridge_node
        
        G.add_edge(prev_node, city2)
        G.add_edge(city2, prev_node)  
     
        speed = random.choice([60, 80, 120])
        capacity = random.choice([500, 700, 800])
        length = random.randint(2, 6)

        G[prev_node][city2]['speed'] = speed
        G[prev_node][city2]['capacity'] = capacity
        G[prev_node][city2]['length'] = length
        G[prev_node][city2]['Time'] = length/speed
        G[city2][prev_node]['speed'] = speed
        G[city2][prev_node]['capacity'] = capacity
        G[city2][prev_node]['length'] = length
        G[city2][prev_node]['Time'] = length/speed

pos = nx.spring_layout(G, seed=39, weight='length') 
node_colors = ["red" if G.nodes[node].get("Depot", 0) == 1 else "green" if node.startswith("C") else "none" for node in G.nodes()]

labels = {node: node[1:] if node.startswith("C") else "" for node in G.nodes()}
plt.figure(figsize=(12, 7))
nx.draw(G, pos, with_labels=True, node_color=node_colors, node_size=700, font_size=13)
nx.draw_networkx_labels(G, pos, labels, font_size=12, font_color="white") 
plt.show()

print(f"Number of Bridges {len([node for node in G.nodes() if node.startswith('B')])}")

for node in G.nodes():
    if "B" in node :
        print("")
        print(f"Time Window of {node} is {G.nodes[node]['Start']} to {G.nodes[node]['Due']}")
        print(f"BFI is {G.nodes[node]['BFI']}")
        for team in Teams:
            print(f"Cost of reparing bridge {node} by team {team} is $ {G.nodes[node]['cost'][team]*1000}")
            print(f"The New BFI of bridge {node} by team {team} would be {G.nodes[node]['NewBFI'][team]}")
Bridges_Count = len([node for node in G.nodes() if node.startswith('B')])
print(sum(
    G.nodes[i]['BFI']
    for i in G.nodes() if i.startswith("B")
) / Bridges_Count)

In [ ]:

import networkx as nx

def compute_shortest_paths(G):
    shortest_paths = {}
    for source in G.nodes:
        for target in G.nodes:
            if source != target:
                try:
                    path = nx.shortest_path(G, source=source, target=target, weight='Time')
                    time = nx.shortest_path_length(G, source=source, target=target, weight='Time')
                    shortest_paths[(source, target)] = (path, time)
                except nx.NetworkXNoPath:
                    shortest_paths[(source, target)] = (None, float('inf'))
    return shortest_paths

shortest_paths = compute_shortest_paths(G)

#-----------------------------%%%-------------------------
import gurobipy as gp
from gurobipy import GRB, quicksum

PlanningHorizon = 8
M = 1000
Bridges_Count = len([node for node in G.nodes() if node.startswith('B')])
m = gp.Model('BIM')

Depots = [d for d in G.nodes() if G.nodes[d].get("Depot") == 1]
pair_dk = [(d, k) for d in Depots for k in Teams]

x = {
    (i, j, dk): m.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_{dk[0]}_{dk[1]}")
    for i in G.nodes() for j in G.nodes() if i != j
    for dk in pair_dk
}

y = {
    (i, dk, t): m.addVar(vtype=GRB.BINARY, name=f"y_{i}_{dk[0]}_{dk[1]}_{t}")
    for i in G.nodes() if i.startswith("B")
    for dk in pair_dk
    for t in range(PlanningHorizon)
}

s = {
    (i, dk): m.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"s_{i}_{dk[0]}_{dk[1]}")
    for i in G.nodes() if i.startswith("B")
    for dk in pair_dk
}


#Each Bridge must be serviced at most once

m.addConstrs((
    quicksum(y[i, dk, t] for dk in pair_dk for t in range(PlanningHorizon)) <= 1
    for i in G.nodes() if i.startswith("B")),name='Served_ONCE'
)


# Linking x and y

m.addConstrs(
    (quicksum(x[i, j, dk] for i in G.nodes() if i != j and (i.startswith("B") or i == dk[0])) ==
    quicksum(y[j, dk, t] for t in range(PlanningHorizon))
    for dk in pair_dk for j in G.nodes() if j.startswith("B")),name="Linki"
)

m.addConstrs(
    (quicksum(x[i, j, dk] for j in G.nodes() if j != i and (j.startswith("B") or j == dk[0])) ==
    quicksum(y[i, dk, t] for t in range(PlanningHorizon))
    for dk in pair_dk for i in G.nodes() if i.startswith("B")),name="Linkj"
)


#Depot Departure and return

m.addConstrs(
    (quicksum(x[dk[0], j, dk] for j in G.nodes() if j.startswith("B")) == 1
    for dk in pair_dk),name='Depot_dk_'
)

m.addConstrs(
    (quicksum(x[i, dk[0], dk] for i in G.nodes() if i.startswith("B")) == 1
    for dk in pair_dk),name='Return_dk'
)


#Time Window

m.addConstrs(
    (s[i, dk] + ServiceTime[dk[1]] - t <= M * (1 - y[i, dk, t])
    for i in G.nodes() if i.startswith('B')
    for dk in pair_dk
    for t in range(PlanningHorizon)),name='y_s_U'
)

m.write('BRIDGE.lp')
m.addConstrs(
    (s[i, dk] + ServiceTime[dk[1]] - t >= -M * (1 - y[i, dk, t])
    for i in G.nodes() if i.startswith('B')
    for dk in pair_dk
    for t in range(PlanningHorizon)),name='y_s_L'
)

m.addConstrs(
    (s[j, dk] >= (s[i, dk] if i.startswith('B') else 0) + ServiceTime[dk[1]] + shortest_paths[i,j][1]/24 - M * (1 - x[i, j, dk])
    for i in G.nodes() for j in G.nodes() if i != j and j.startswith("B")
    for dk in pair_dk
    if i.startswith("B") or G.nodes[i].get("Depot")),name='s_Start'
)

# m.addConstrs(
#     (s[i,dk] >= G.nodes[i]['Start'] for i in G.nodes() if i.startswith('B') for dk in pair_dk)
# )

m.addConstrs(
    (s[i, dk] + ServiceTime[dk[1]] <= G.nodes[i]['Due']
    for i in G.nodes() if i.startswith("B")
    for dk in pair_dk),name='Due'
)

Earlyvisiting = quicksum(s[i,dk] for i in G.nodes() if i.startswith('B') for dk in pair_dk)


Resilience_Raw = quicksum(
    G.nodes[i]['BFI'] * (1- quicksum(y[i, dk, t] for dk in pair_dk for t in range(PlanningHorizon)))
    for i in G.nodes() if i.startswith('B')
)
Resilience = quicksum(
    G.nodes[i]['NewBFI'][dk[1]] * quicksum(y[i, dk, t] for t in range(PlanningHorizon))
    for i in G.nodes() if i.startswith("B")
    for dk in pair_dk
)

Cost = quicksum(
    G.nodes[i]['cost'][dk[1]] * quicksum(y[i, dk, t] for t in range(PlanningHorizon))
    for i in G.nodes() if i.startswith("B")
    for dk in pair_dk
)

VisitedCount = quicksum(y[i, dk, t] for i in G.nodes() if i.startswith('B') for dk in pair_dk for t in range(PlanningHorizon))

# m.addConstr((Resilience + Resilience_Raw)/Bridges_Count >= 0.75)
# m.addConstr((Resilience + Resilience_Raw)/Bridges_Count <= 0.76)
m.write('BRIDGE.lp')
m.setObjective((Resilience + Resilience_Raw)/Bridges_Count ,GRB.MAXIMIZE)
# m.setObjective(Cost,GRB.MINIMIZE)
# m.Params.OutputFlag = 0
# m.Params.MIPGap = 0.04

m.optimize()

if m.Status == GRB.INFEASIBLE:
    m.computeIIS()
    for c in m.getConstrs():
        if c.IISConstr:
            print(f" - {c.ConstrName}")

count = 0
if m.Status != GRB.INFEASIBLE:
    print(f"Cost is {quicksum(
    G.nodes[i]['cost'][dk[1]] * quicksum(y[i, dk, t].X for t in range(PlanningHorizon))
    for i in G.nodes() if i.startswith("B")
    for dk in pair_dk
)} with Resilience of {(quicksum(
    G.nodes[i]['BFI'] * (1- quicksum(y[i, dk, t].X for dk in pair_dk for t in range(PlanningHorizon)))
    for i in G.nodes() if i.startswith('B')
) + quicksum(
    G.nodes[i]['NewBFI'][dk[1]] * quicksum(y[i, dk, t].X for t in range(PlanningHorizon))
    for i in G.nodes() if i.startswith("B")
    for dk in pair_dk
))/Bridges_Count } ")
    active_edges = []
    for var in m.getVars():
        if var.X >0:
            print(f" {var.VarName} is {var.X}")
            if "y" in var.VarName:
                count +=1
        if var.X >0 and "x" in var.VarName:
            active_edges.append(var.VarName)

print(f"Number of visited Bridges {count}")
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import Patch

team_colors = {
    "RRU": "blue",
    "ERT": "orange",
    "CIRS": "green"
}


edge_list_by_team = {"RRU": [], "ERT": [], "CIRS": []}

for (i, j, dk), var in x.items():
    if var.X > 0.5:
        d, k = dk
        edge_list_by_team[k].append((i, j))


plt.figure(figsize=(12, 7))
nx.draw(
    G, pos, with_labels=True, node_color=node_colors,
    node_size=700, font_size=10, edge_color="lightgray"
)


for team, edges in edge_list_by_team.items():
    nx.draw_networkx_edges(
        G, pos, edgelist=edges, width=2, edge_color=team_colors[team], label=team
    )

legend_elements = [
    Patch(facecolor=color, edgecolor='black', label=team)
    for team, color in team_colors.items()
]
plt.legend(handles=legend_elements, title="Teams", loc="lower right")

plt.title("Emergency Routing Paths by Team", fontsize=14)
plt.axis("off")
plt.show()



## Gantt

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict


team_colors = {
    "RRU": "skyblue",
    "ERT": "salmon",
    "CIRS": "lightgreen"
}

schedule_data = []

for var in m.getVars():
    if 's' in var.VarName:
        if var.X > 0.5:  
            varname = var.VarName 
            parts = varname.split("_")
            if len(parts) == 4 and parts[0] == 's':
                _, bridge, depot, team = parts
                start_time = var.X
                schedule_data.append((bridge, team, depot, start_time))


grouped_schedule = defaultdict(list)
for bridge, team, depot, start in schedule_data:
    grouped_schedule[depot].append((bridge, team, start))


fig, axes = plt.subplots(nrows=len(grouped_schedule), figsize=(10, 3 * len(grouped_schedule)), sharex=True)


if len(grouped_schedule) == 1:
    axes = [axes]

for ax, (depot, tasks) in zip(axes, grouped_schedule.items()):
    yticks = []
    yticklabels = []
    for i, (bridge, team, start) in enumerate(tasks):
        end = start + 1 
        ax.barh(i, 1, left=start, height=0.6, color=team_colors.get(team, 'gray'))
        ax.text(start + 0.05, i, f"{bridge} ({team})", va='center', ha='left', fontsize=8)
        yticks.append(i)
        yticklabels.append(bridge)

    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels)
    ax.set_title(f"Gantt Chart - Depot {depot}")
    ax.invert_yaxis()
    ax.grid(axis='x', linestyle='--', alpha=0.5)

plt.xlabel("Time")
plt.tight_layout()
plt.show()


### Epsilon

In [ ]:
import numpy as np
Resilience_array = []
Cost_array = []

def Pareto_Frontier(m, flag):
    if flag == 1:
        m.setObjective((Resilience + Resilience_Raw)/Bridges_Count,GRB.MAXIMIZE)
        # m.Params.OutPutFlag = 0
        m.optimize()
        COST = quicksum(
            G.nodes[i]['cost'][dk[1]] * quicksum(y[i, dk, t].X for t in range(PlanningHorizon))
            for i in G.nodes() if i.startswith("B")
            for dk in pair_dk
                        )
        return m.ObjVal, COST.getValue()
    elif flag == 2: 
        m.setObjective(Cost,GRB.MINIMIZE)
        # m.Params.OutPutFlag = 0
        m.optimize()
        RES = (quicksum(
                G.nodes[i]['BFI'] * (1- quicksum(y[i, dk, t].X for dk in pair_dk for t in range(PlanningHorizon)))
                for i in G.nodes() if i.startswith('B')
            ) + quicksum(
                G.nodes[i]['NewBFI'][dk[1]] * quicksum(y[i, dk, t].X for t in range(PlanningHorizon))
                for i in G.nodes() if i.startswith("B")
                for dk in pair_dk
            ))/Bridges_Count
        return m.ObjVal, RES.getValue()
    
MAX_Resilience, MAX_COST = Pareto_Frontier(m,flag=1)
MIN_COST, MIN_Resilience = Pareto_Frontier(m,flag=2)


print(f"MIN value for Resilience is {MIN_Resilience} and Min value for cost is {MIN_COST}")
print(f"MAX value for Resilience is {MAX_Resilience} and MAX value for cost is {MAX_COST}")

NUM_EPSILONS = 10
EPSILON = np.linspace(MIN_COST, MAX_COST, NUM_EPSILONS)

for eps in EPSILON:
    m.addConstr(Cost <= eps,name='epsilon_constraint')
    MAX_Resilience, MAX_COST= Pareto_Frontier(m,flag=1)
    Resilience_array.append(MAX_Resilience)
    Cost_array.append(MAX_COST)
    constr = m.getConstrByName("epsilon_constraint")
    m.remove(constr)
    m.update()

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))


plt.scatter(Resilience_array, Cost_array, 
            c='blue', edgecolors='black', s=70, marker='o', alpha=0.8)


plt.title("Pareto Frontier: Cost vs. Resilience", fontsize=14, fontweight='bold')
plt.xlabel("Resilience", fontsize=12, color='blue')
plt.ylabel("Cost ($1000)", fontsize=12, color='red')
plt.tick_params(axis='y', colors='red')
plt.tick_params(axis='x', colors='blue')


for x, y in zip(Resilience_array, Cost_array):
    plt.text(x, y + 0.5, f"{y:.0f}", fontsize=9, color='red', ha='center')  # Cost بالای نقطه
    plt.text(x, y - 1.5, f"{x:.2f}", fontsize=9, color='blue', ha='center')   # Resilience پایین نقطه


plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()

